# Step 4: Choose and tune the final model, then test once on 2026
1. Feature set = features the random forest kept in at least 3 of the 4 folds (chosen on 2021-2025 only).
2. Tune LightGBM and random-forest classifiers on the forward folds; also try their average. Choose by mean log-loss.
3. Fit a smoothed wicket-cost model on 2021-2025.
4. Lock everything, then score 2026 **once**.

In [1]:
import os, json, warnings; warnings.filterwarnings('ignore'); ROOT=os.path.expanduser('~/hundred-momentum')
import numpy as np, pandas as pd, lightgbm as lgb, optuna, joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import log_loss, roc_auc_score
optuna.logging.set_verbosity(optuna.logging.WARNING)
feats_all = json.load(open(ROOT+'/data/clean/feature_list.json'))
ALLD = pd.read_parquet(ROOT+'/data/clean/features.parquet')
ALLD['cls'] = np.where(ALLD.is_wicket == 1, 0, 1 + np.minimum(ALLD.runs_total, 5))
D = ALLD[(ALLD.season <= 2025) & (ALLD.full_match == 1)].reset_index(drop=True)      # development data only
cv = pd.read_csv(ROOT+'/data/clean/cv_results.csv')

## 4.1 Feature set
Re-derived from the same per-fold permutation importances as Step 3 (saved there would be identical; recomputed here for a self-contained notebook is slow, so the Step 3 rule is applied to its printed selections).

In [2]:
import nbformat
nb = nbformat.read(ROOT+'/notebooks/03_regressor_vs_classifier.ipynb', 4)
# rebuild selections exactly as in Step 3 by re-running its selection function on the development data
exec(nb.cells[1].source.replace("D = pd.read_parquet(ROOT+'/data/clean/features.parquet'); D = D[(D.season <= 2025) & (D.full_match == 1)].reset_index(drop=True); assert D.season.max() == 2025", "D = D"))
exec(nb.cells[3].source.split('SEL, IMP')[0])
SEL = [[f for f, x, y in zip(feats, *perm_select(tr)) if x > 0 or y > 0] for tr, va, s in FOLDS]
cnt = pd.Series([f for s in SEL for f in s]).value_counts(); FEATS = sorted(cnt[cnt >= 3].index); print(len(FEATS), 'features'); print(FEATS)
json.dump(FEATS, open(ROOT+'/data/clean/final_features.json','w'))

(30846, 106) {'W': np.int64(1907), '0': np.int64(8026), '1': np.int64(12865), '2': np.int64(2474), '3': np.int64(171), '4': np.int64(3624), '5+': np.int64(1779)}


42 features
['ball_no', 'balls_left', 'bat_balls', 'bat_hist_balls', 'bat_hist_bnd', 'bat_hist_runs', 'bat_runs', 'bat_sr10', 'batph_hist_balls', 'batph_hist_dot', 'batph_hist_runs', 'bowl_balls', 'bowl_dots10', 'bowl_econ', 'bowl_hist_dot', 'bowl_hist_wkt', 'bowl_wkts', 'bowlph_hist_balls', 'bowlph_hist_bnd', 'chase_target', 'form_vs_hist', 'free_hit', 'innings2', 'last10_wkt', 'last20_runs', 'last20_wkt', 'last5_runs', 'last5_wkt', 'matchup_runs', 'part_balls', 'part_runs', 'powerplay', 'prev_runs', 'rate_gap', 'req_rate', 'runs_needed', 'score_pre', 'season_hist_bnd', 'season_hist_dot', 'since_wicket', 'venue_hist_bnd', 'wkts_per_ball_left']


## 4.2 Tune on the forward folds

In [3]:
X = D[FEATS].astype(float); y = D.cls.values; gid = D.match_id.values; S = D.season.values
FOLDS = [(np.where(S < s)[0], np.where(S == s)[0], s) for s in (2022, 2023, 2024, 2025)]
def inner(tr): a, b = next(GroupShuffleSplit(1, test_size=.2, random_state=7).split(tr, groups=gid[tr])); return tr[a], tr[b]
def full(P, lab): Q = np.full((len(P), 7), 1e-9); Q[:, lab] = P; return Q / Q.sum(1, keepdims=True)
def fit_lgb(p, tr):
    a, b = inner(tr); return lgb.LGBMClassifier(**p).fit(X.iloc[a], y[a], eval_set=[(X.iloc[b], y[b])], callbacks=[lgb.early_stopping(80, verbose=False)])
def lgb_params(t): return dict(n_estimators=3000, learning_rate=t.suggest_float('learning_rate', .005, .08, log=True), num_leaves=t.suggest_int('num_leaves', 4, 48), max_depth=t.suggest_int('max_depth', 2, 8),
    min_child_samples=t.suggest_int('min_child_samples', 40, 600, log=True), subsample=t.suggest_float('subsample', .5, 1), subsample_freq=1, colsample_bytree=t.suggest_float('colsample_bytree', .2, 1),
    reg_lambda=t.suggest_float('reg_lambda', 1e-3, 50, log=True), min_split_gain=t.suggest_float('min_split_gain', 0, .2), random_state=42, n_jobs=-1, verbose=-1)
def cv_lgb(p): 
    out = []
    for tr, va, s in FOLDS: mdl = fit_lgb(p, tr); out.append((log_loss(y[va], full(mdl.predict_proba(X.iloc[va]), mdl.classes_), labels=range(7)), mdl.best_iteration_))
    return out
st = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
st.optimize(lambda t: np.mean([a for a, _ in cv_lgb(lgb_params(t))]), n_trials=40)
P_LGB = lgb_params(optuna.trial.FixedTrial(st.best_params)); print('LightGBM best CV log-loss', round(st.best_value, 5), st.best_params)
def rf_cv(leaf, mf):
    r = []
    for tr, va, s in FOLDS:
        med = X.iloc[tr].median(); mdl = RandomForestClassifier(n_estimators=500, min_samples_leaf=leaf, max_features=mf, n_jobs=-1, random_state=1).fit(X.iloc[tr].fillna(med), y[tr])
        r.append(full(mdl.predict_proba(X.iloc[va].fillna(med)), mdl.classes_))
    return r
grid = {(l, f): rf_cv(l, f) for l in (25, 60, 120) for f in (.2, .4)}
ll = lambda Ps: np.mean([log_loss(y[va], P, labels=range(7)) for P, (tr, va, s) in zip(Ps, FOLDS)])
best_rf = min(grid, key=lambda k: ll(grid[k])); print('RF grid:', {k: round(ll(v), 5) for k, v in grid.items()}, '-> best', best_rf)
PL = []
for tr, va, s in FOLDS: mdl = fit_lgb(P_LGB, tr); PL.append(full(mdl.predict_proba(X.iloc[va]), mdl.classes_))
cand = {'LightGBM (tuned)': PL, 'Random forest (tuned)': grid[best_rf], 'Average of both': [(a + b) / 2 for a, b in zip(PL, grid[best_rf])]}
base = cv[(cv.features == 'selected') & (cv.model.isin(['Baseline: phase x wickets', 'Clf: LightGBM', 'Clf: random forest']))].groupby('model').logloss.mean()
res = pd.DataFrame({k: [log_loss(y[va], P, labels=range(7)) for P, (tr, va, s) in zip(v, FOLDS)] for k, v in cand.items()}, index=[2022, 2023, 2024, 2025]).T
res['mean'] = res.mean(1); print(res.round(5)); print('untuned, Step 3:', base.round(5).to_dict())
WIN = res['mean'].idxmin(); print('CHOSEN:', WIN)

LightGBM best CV log-loss 1.49762 {'learning_rate': 0.009693667964438477, 'num_leaves': 18, 'max_depth': 2, 'min_child_samples': 122, 'subsample': 0.6102333982607775, 'colsample_bytree': 0.6142283270654846, 'reg_lambda': 11.918375705508595, 'min_split_gain': 0.10268276595271711}


RF grid: {(25, 0.2): np.float64(1.49699), (25, 0.4): np.float64(1.49736), (60, 0.2): np.float64(1.49576), (60, 0.4): np.float64(1.49593), (120, 0.2): np.float64(1.49605), (120, 0.4): np.float64(1.49555)} -> best (120, 0.4)


                          2022     2023     2024     2025     mean
LightGBM (tuned)       1.50589  1.50971  1.48537  1.48950  1.49762
Random forest (tuned)  1.50111  1.50725  1.48577  1.48809  1.49555
Average of both        1.50215  1.50684  1.48425  1.48726  1.49513
untuned, Step 3: {'Baseline: phase x wickets': 1.51385, 'Clf: LightGBM': 1.49955, 'Clf: random forest': 1.49689}
CHOSEN: Average of both


## 4.3 Smoothed wicket cost
Expected runs still to come in a first innings, as a function of balls left and wickets in hand, fitted with monotone constraints (more balls or more wickets can never mean fewer runs). Cost of a wicket = E(balls, wickets in hand) - E(balls, one fewer wicket), always >= 0.

In [4]:
F1 = D[(D.innings == 1)].copy(); F1['rest'] = F1.match_id.map(F1.groupby('match_id').runs_total.sum()) - F1.score_pre; F1['wih'] = 10 - F1.wkts_pre
RR = lgb.LGBMRegressor(n_estimators=400, learning_rate=.03, num_leaves=8, min_child_samples=200, monotone_constraints=[1, 1], random_state=0, verbose=-1).fit(F1[['balls_left', 'wih']], F1.rest)
E = lambda b, w: RR.predict(pd.DataFrame({'balls_left': np.asarray(b, float), 'wih': np.asarray(w, float)}))
def wicket_cost(balls_left, wkts_pre):
    b = np.maximum(np.asarray(balls_left) - 1, 0); w = 10 - np.asarray(wkts_pre)
    return np.where(w - 1 <= 0, E(b, w), E(b, w) - E(b, np.maximum(w - 1, 0)))
for bl, wp in [(70, 1), (50, 3), (30, 3), (10, 3), (10, 5)]: print(f'balls left {bl:>3}, wickets down {wp}: expected runs left {E([bl],[10-wp])[0]:6.1f}, wicket cost {wicket_cost([bl],[wp])[0]:5.2f}')
grid_b, grid_w = np.meshgrid(np.arange(1, 101), np.arange(0, 10)); wc = wicket_cost(grid_b.ravel(), grid_w.ravel()); print('min cost over all states:', round(wc.min(), 3), '| max:', round(wc.max(), 2))
joblib.dump(RR, ROOT+'/data/clean/runs_left_model.pkl')

balls left  70, wickets down 1: expected runs left  108.1, wicket cost  8.90
balls left  50, wickets down 3: expected runs left   75.5, wicket cost  6.95
balls left  30, wickets down 3: expected runs left   54.9, wicket cost  8.38
balls left  10, wickets down 3: expected runs left   21.7, wicket cost  1.84
balls left  10, wickets down 5: expected runs left   18.1, wicket cost  0.00
min cost over all states: 0.0 | max: 117.72


['/Users/mashadniyazi/hundred-momentum/data/clean/runs_left_model.pkl']

## 4.4 Lock, then test once on 2026
Final model is trained on all of 2021-2025. Baselines are also trained on 2021-2025 only.

In [5]:
locked = dict(model=WIN, features=FEATS, lgb=P_LGB, rf=dict(min_samples_leaf=best_rf[0], max_features=best_rf[1]))
json.dump(locked, open(ROOT+'/data/clean/locked_model.json','w'), indent=1, default=str); print('locked:', WIN)
tr_all = np.arange(len(D)); med = X.median(); models = {}
if WIN != 'Random forest (tuned)': models['lgb'] = fit_lgb(P_LGB, tr_all)
if WIN != 'LightGBM (tuned)': models['rf'] = RandomForestClassifier(n_estimators=500, min_samples_leaf=best_rf[0], max_features=best_rf[1], n_jobs=-1, random_state=1).fit(X.fillna(med), y)
def predict(Z):
    Ps = []
    if 'lgb' in models: Ps.append(full(models['lgb'].predict_proba(Z[FEATS].astype(float)), models['lgb'].classes_))
    if 'rf' in models: Ps.append(full(models['rf'].predict_proba(Z[FEATS].astype(float).fillna(med)), models['rf'].classes_))
    return sum(Ps) / len(Ps)
joblib.dump(dict(models=models, med=med, feats=FEATS), ROOT+'/data/clean/final_model.pkl')
# ---------------- 2026: opened here for the first time ----------------
T = ALLD[(ALLD.season == 2026) & (ALLD.full_match == 1)].reset_index(drop=True); yt = T.cls.values; P = predict(T)
cm = np.array([D.runs_total[D.cls == c].mean() for c in range(7)]); er = P @ cm
pr = np.bincount(y, minlength=7) / len(y); P0 = np.tile(pr, (len(T), 1))
key = lambda d: (np.minimum(d.ball_no - 1, 99) // 25 * 10 + np.minimum(d.wkts_pre, 6)).values
tb = pd.DataFrame(np.eye(7)[y]).groupby(key(D)).sum().add(50 * pr, axis=1); tb = tb.div(tb.sum(1), axis=0); PB = np.vstack([tb.loc[k].values if k in tb.index else pr for k in key(T)])
O = np.eye(7)[yt]; rt = T.runs_total.values
row = lambda nm, Q: dict(model=nm, logloss=log_loss(yt, Q, labels=range(7)), brier=((Q - O) ** 2).sum(1).mean(), R2_runs=1 - ((rt - Q @ cm) ** 2).sum() / ((rt - D.runs_total.mean()) ** 2).sum(),
    AUC_wicket=roc_auc_score(yt == 0, Q[:, 0]), AUC_boundary=roc_auc_score(T.runs_bat.values >= 4, Q[:, 5:].sum(1)), AUC_dot=roc_auc_score(yt == 1, Q[:, 1]))
TEST = pd.DataFrame([row('Baseline: mean', P0), row('Baseline: phase x wickets', PB), row(WIN, P)]); print('2026 TEST', len(T), 'balls,', T.match_id.nunique(), 'matches'); print(TEST.round(4).to_string(index=False))
TEST.to_csv(ROOT+'/data/clean/test_2026.csv', index=False)
mid = T.match_id.values; um = np.unique(mid); pos = {q: np.where(mid == q)[0] for q in um}; rng = np.random.default_rng(1)
lm = -np.log(P[np.arange(len(T)), yt]); lb = -np.log(PB[np.arange(len(T)), yt]); l0 = -np.log(P0[np.arange(len(T)), yt])
bs = lambda d: np.percentile([d[np.concatenate([pos[q] for q in rng.choice(um, len(um))])].mean() for _ in range(2000)], [2.5, 97.5])
print('log-loss gain v mean baseline:', round((l0 - lm).mean(), 5), bs(l0 - lm).round(5), '| v phase x wickets:', round((lb - lm).mean(), 5), bs(lb - lm).round(5))
for nm, p, a in [('wicket', P[:, 0], yt == 0), ('boundary', P[:, 5:].sum(1), T.runs_bat.values >= 4)]:
    q = pd.qcut(p, 5, labels=False); print(nm, 'predicted', np.round([p[q == k].mean() for k in range(5)], 3), 'actual', np.round([a[q == k].mean() for k in range(5)], 3))

locked: Average of both


2026 TEST 6834 balls, 34 matches
                    model  logloss  brier  R2_runs  AUC_wicket  AUC_boundary  AUC_dot
           Baseline: mean   1.5394 0.7342   0.0000      0.5000        0.5000   0.5000
Baseline: phase x wickets   1.5108 0.7214   0.0082      0.6085        0.5646   0.6020
          Average of both   1.4901 0.7140   0.0288      0.6192        0.6113   0.6352
log-loss gain v mean baseline: 0.04923 [0.0402 0.0592] | v phase x wickets: 0.0207 [0.0163  0.02514]
wicket predicted [0.042 0.052 0.058 0.067 0.102] actual [0.037 0.038 0.047 0.059 0.102]
boundary predicted [0.114 0.155 0.182 0.218 0.262] actual [0.111 0.154 0.206 0.222 0.285]


## 4.5 Save out-of-sample probabilities for the momentum step
2022-2025 from the forward-fold models (each trained only on earlier seasons); 2026 from the final model. 2021 has no out-of-sample prediction.

In [6]:
rows = []
for (tr, va, s) in FOLDS:
    sub = {}
    if WIN != 'Random forest (tuned)': mdl = fit_lgb(P_LGB, tr); sub['l'] = full(mdl.predict_proba(X.iloc[va]), mdl.classes_)
    if WIN != 'LightGBM (tuned)': m2 = RandomForestClassifier(n_estimators=500, min_samples_leaf=best_rf[0], max_features=best_rf[1], n_jobs=-1, random_state=1).fit(X.iloc[tr].fillna(X.iloc[tr].median()), y[tr]); sub['r'] = full(m2.predict_proba(X.iloc[va].fillna(X.iloc[tr].median())), m2.classes_)
    Q = sum(sub.values()) / len(sub); rows.append(D.iloc[va].assign(**{f'p_{k}': Q[:, i] for i, k in enumerate(['W','0','1','2','3','4','5p'])}))
rows.append(T.assign(**{f'p_{k}': P[:, i] for i, k in enumerate(['W','0','1','2','3','4','5p'])}))
OUT = pd.concat(rows, ignore_index=True); OUT['exp_runs'] = OUT[[f'p_{k}' for k in ['W','0','1','2','3','4','5p']]].values @ cm
OUT['wicket_cost'] = wicket_cost(OUT.balls_left, OUT.wkts_pre); OUT.to_parquet(ROOT+'/data/clean/predictions.parquet', index=False); print(OUT.shape, OUT.season.value_counts().sort_index().to_dict())

(31732, 115) {2022: 6826, 2023: 5652, 2024: 5896, 2025: 6524, 2026: 6834}
